# 63. The `DalitzAmplitude` wrapper

**Objectives:**

- Wrap a two-dimensional `QMI2D` amplitude in `DalitzAmplitude` inside a `DecayModel`,
  alongside an ordinary one-dimensional `Resonance`.
- Look at `DalitzAmplitude`'s own three fields -- `dynamics`, `coefficient`,
  `normalize_component` -- rather than `QMI2D`'s interpolation modes
  ([tutorial 10](tutorial_10_qmi2d_dalitz_closure.ipynb) already covers those).
- Evaluate the coherent sum of the two components and confirm each contributes as
  its own dynamics/coefficient prescribe.

Run cells top to bottom in a fresh kernel. Masses are in GeV, invariants in GeV²,
daughter indices start at zero.

In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede any numerical work: amplitudes use complex128.

import jax.numpy as jnp
import numpy as np

from dalitzplotfitter import (
    DalitzAmplitude, DecayChannel, DecayModel, Parameter, QMI2D, RealImag,
    Resonance, generate_toy,
)

## 1. Why `DalitzAmplitude` exists

`Resonance` composes a lineshape through the ordinary `lineshape(mass, context)`
interface: it is intrinsically one-dimensional, a function of one resonance-pair
invariant mass, combined with an angular factor and barriers (CLAUDE.md,
"One-dimensional lineshapes vs. full 2D Dalitz amplitudes"). `QMI2D` is instead a
complex field defined directly over `(s12, s13)` -- there is no single "resonance
mass" it is a function of. `DalitzAmplitude` bypasses the one-dimensional isobar
construction entirely so a component like this can sit in the same coherent sum as
ordinary resonances: it needs only `name`, `dynamics` (any callable
`dynamics(data, parameters) -> Array`), `coefficient`, and the same
`normalize_component` override every other component type has.

In [2]:
channel = DecayChannel("B+", ("K-", "pi+", "pi-"))
m1, m2, m3 = channel.daughter_masses
smin = (m1 + m2) ** 2
smax = (channel.parent_mass - m3) ** 2
edges = tuple(np.linspace(smin, smax, 4))  # 3x3 cells, unfolded: no identical daughters here

magnitudes = tuple(tuple(0.8 for _ in range(3)) for _ in range(3))
phases = (
    (0.0, 0.3, 0.6),
    (0.3, 0.6, 0.9),
    (0.6, 0.9, 1.2),
)
field = QMI2D(
    s12_edges=edges, s13_edges=edges,
    magnitudes=magnitudes, phases=phases,
    interpolation="linear", folded=False,
)
print("QMI2D field built over", (len(edges) - 1), "x", (len(edges) - 1), "cells")

QMI2D field built over 3 x 3 cells


## 2. One `DalitzAmplitude` component alongside one `Resonance`

Both are ordinary entries in `DecayModel.components`; `DalitzAmplitude` needs no
`pair`, `mass`, `width` or `spin` -- `QMI2D` already reads `s12`/`s13` off the event
data directly.

In [3]:
model = DecayModel(
    channel,
    [Resonance("Kstar", (0, 1), RealImag(1.0, 0.0), mass=0.8958, width=0.0474, spin=1),
     DalitzAmplitude("qmi2d", field, RealImag(0.6, 0.2))],
    normalize_components=False,
    normalization_method="square-dalitz", normalization_pair=(0, 1),
    normalization_resolution=50,
)
truth = {p.name: p.value for p in model.parameters}
data = generate_toy(
    model, 1000, parameters=truth, seed=63,
    method="inverse-transform", inverse_resolution=256, include_momenta=False,
)
print(f"Generated {data.size} events from the coherent Kstar + QMI2D model")

Generated 1000 events from the coherent Kstar + QMI2D model


## 3. The coherent sum, component by component

`DecayModel.amplitude` sums every component's `coefficient * scale * dynamics(data)`
(`decay.py`'s `DecayModel.amplitude`). With `normalize_components=False` and no
per-component override, `scale=1` for both, so the total is exactly
`c_Kstar * F_Kstar(data) + c_qmi2d * field(data)` -- confirming `DalitzAmplitude`
contributes its raw `dynamics` output through the same coefficient mechanism as any
other component, with no isobar machinery (angular factor, barrier, resonance mass)
in between.

In [4]:
data_dict = data.as_dict()
kstar_component = next(
    c for c in model.amplitude_model.components if c.name == "Kstar"
)
qmi2d_component = next(
    c for c in model.amplitude_model.components if c.name == "qmi2d"
)

manual_amplitude = (
    (1.0 + 0.0j) * jnp.asarray(kstar_component.function(data_dict, None))
    + (0.6 + 0.2j) * jnp.asarray(field(data_dict))
)
model_amplitude = model.amplitude(data_dict, truth)
np.testing.assert_allclose(manual_amplitude, model_amplitude, atol=1e-10)

model_intensity = model.intensity(data_dict, truth)
np.testing.assert_allclose(jnp.abs(model_amplitude) ** 2, model_intensity, atol=1e-10)
print("Coherent sum = Kstar's own coefficient*dynamics + qmi2d's own coefficient*dynamics.")

Coherent sum = Kstar's own coefficient*dynamics + qmi2d's own coefficient*dynamics.


## 4. `normalize_component` still applies to a `DalitzAmplitude`

Setting `normalize_component=True` on the `qmi2d` component rescales its own raw
`field(data)` output to unit integral before the coefficient is applied -- exactly
the same per-component convention `Resonance(..., normalize_component=True)` uses
(CLAUDE.md, "Normalization: the central invariant"), even though `field` here is a
2D `QMI2D` amplitude rather than a 1D lineshape.

In [5]:
normalized_model = DecayModel(
    channel,
    [Resonance("Kstar", (0, 1), RealImag(1.0, 0.0), mass=0.8958, width=0.0474, spin=1),
     DalitzAmplitude("qmi2d", field, RealImag(0.6, 0.2), normalize_component=True)],
    normalize_components=False,
    normalization_method="square-dalitz", normalization_pair=(0, 1),
    normalization_resolution=50,
)
norm_sample = normalized_model.normalization_sample
raw_field_values = jnp.asarray(field(norm_sample.as_dict()))
unit_integral = jnp.mean(norm_sample.weights * jnp.abs(raw_field_values) ** 2)
print(f"integral(|field|^2) before rescaling: {float(unit_integral):.4f}")

# `Resonance`/`DalitzAmplitude` both rescale by 1/sqrt(integral |F|^2 dPhi) when
# normalize_component is True (CLAUDE.md, "Normalization: the central invariant").
scale = 1.0 / jnp.sqrt(unit_integral)
scaled_integral = jnp.mean(norm_sample.weights * jnp.abs(scale * raw_field_values) ** 2)
np.testing.assert_allclose(scaled_integral, 1.0, atol=1e-8)

normalized_amplitude = normalized_model.amplitude(norm_sample.as_dict(), {})
manual_scaled_qmi2d = scale * raw_field_values
manual_kstar_norm = jnp.asarray(
    next(c for c in normalized_model.amplitude_model.components if c.name == "Kstar")
    .function(norm_sample.as_dict(), None)
)
np.testing.assert_allclose(
    normalized_amplitude,
    (1.0 + 0.0j) * manual_kstar_norm + (0.6 + 0.2j) * manual_scaled_qmi2d,
    atol=1e-9,
)
print("With normalize_component=True, the qmi2d component's own dynamics integrate to 1,")
print("and DecayModel.amplitude applies that same rescaling before the coefficient.")

integral(|field|^2) before rescaling: 223.0105


With normalize_component=True, the qmi2d component's own dynamics integrate to 1,
and DecayModel.amplitude applies that same rescaling before the coefficient.


## Continue learning

[Tutorial 10](tutorial_10_qmi2d_dalitz_closure.ipynb) covers `QMI2D`'s
`none`/`linear`/`cubic`/`hermite` interpolation modes, folding for identical
daughters, and a full fit of a floating node. See
[`docs/lineshapes.md`](../../docs/lineshapes.md#qmi2d-dalitz-amplitude) and
CLAUDE.md's "One-dimensional lineshapes vs. full 2D Dalitz amplitudes" section.

Return to [the course guide](TUTORIALS.md).